# Banking & Fintech Consumer Complaint Analysis

This notebook analyzes a cleaned consumer complaints dataset with a **banking/fintech** focus.

### Questions answered
- Which **products** and **issues** generate the most complaints?
- What are the most common **company response types**?
- Which **states** show the highest complaint volume?
- How do complaints trend **month-by-month**?
- What is the **timely response rate**?

### Output
Running this notebook will generate:
- PNG charts in `outputs/`
- A cleaned export in `outputs/consumer_complaints_banking_cleaned_from_sample.csv`


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

DATA_PATH = "data/consumer_complaints_banking_sample_20000.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Missing {DATA_PATH}.\n"
        "Upload the sample CSV into the repo 'data/' folder."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)

# Parse dates
df["Date received"] = pd.to_datetime(df["Date received"], errors="coerce")

# Time features
df["Month"] = df["Date received"].dt.to_period("M").astype(str)
df["Year"] = df["Date received"].dt.year

df.head()


## Quick checks


In [ ]:
print("Rows, Columns:", df.shape)
print("\nColumns:")
display(df.columns.to_frame(index=False))


In [ ]:
# Missing values (top 12 columns)
missing = df.isna().mean().sort_values(ascending=False).head(12)
missing.to_frame("missing_rate").style.format({"missing_rate":"{:.1%}"})


## KPI summary (numbers recruiters look for)


In [ ]:
total = len(df)

date_min = df["Date received"].min()
date_max = df["Date received"].max()

top_product = df["Product"].value_counts().idxmax() if "Product" in df.columns else None
top_issue = df["Issue"].value_counts().idxmax() if "Issue" in df.columns else None
top_state = df["State"].value_counts().idxmax() if "State" in df.columns else None
top_response = df["Company response to consumer"].value_counts().idxmax() if "Company response to consumer" in df.columns else None

timely_rate = None
if "Timely response?" in df.columns:
    timely_rate = (df["Timely response?"].astype(str).str.lower() == "yes").mean()

print("Total complaints:", f"{total:,}")
print("Date range:", date_min, "→", date_max)
print("Top product:", top_product)
print("Top issue:", top_issue)
print("Top state:", top_state)
print("Most common response:", top_response)
print("Timely response rate:", f"{timely_rate*100:.1f}%" if timely_rate is not None else "N/A")


## Visualization helpers


In [ ]:
plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12
})

def polish(ax):
    ax.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def save_bar(series, title, xlabel, ylabel, filename, top_n=10):
    s = series.dropna().astype(str).str.strip().value_counts().head(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(s.index, s.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    polish(ax)
    fig.tight_layout()
    out_path = os.path.join("outputs", filename)
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)


## Charts


In [ ]:
# Top products
if "Product" in df.columns:
    save_bar(df["Product"], "Top Products by Complaint Volume (Banking/Fintech)", "Complaints", "Product", "01_top_products.png", top_n=9)

# Top issues
if "Issue" in df.columns:
    save_bar(df["Issue"], "Top Issues by Complaint Volume", "Complaints", "Issue", "02_top_issues.png", top_n=10)

# Response types
if "Company response to consumer" in df.columns:
    save_bar(df["Company response to consumer"], "Top Company Response Types", "Count", "Response type", "03_response_types.png", top_n=10)

# Top states
if "State" in df.columns:
    save_bar(df["State"], "Top States by Complaint Volume", "Complaints", "State", "04_top_states.png", top_n=10)


## Complaints over time (monthly trend)


In [ ]:
trend = df.dropna(subset=["Month"]).groupby("Month").size()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(trend.index, trend.values, marker="o", linewidth=1.5)
ax.set_title("Complaints Over Time (Monthly)")
ax.set_xlabel("Month")
ax.set_ylabel("Complaints")
ax.tick_params(axis="x", rotation=45)
polish(ax)
fig.tight_layout()

out_path = os.path.join("outputs", "05_trend_monthly.png")
fig.savefig(out_path, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_path)


## Save cleaned export (reusable in Excel/SQL)


In [ ]:
# Minimal cleaning: trim key text columns (safe)
for col in ["Product", "Issue", "Sub-issue", "Company response to consumer", "State"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().replace({"nan": np.nan})

clean_out = os.path.join("outputs", "consumer_complaints_banking_cleaned_from_sample.csv")
df.to_csv(clean_out, index=False)
print("Saved:", clean_out)
